In [ ]:
%run _bootstrap_dev.ipynb

## Definizione strategie e tickers

In [ ]:

#################################
# Selezione strategie
#################################

# Tutte le strategie definite 
all_strategies=list_strategies()

kryptera_strategies = ['kc_std_volatility_fade', 'bb_lower_reclaim_piercing', 'gann_cci_kcb_momentum']
# Esempi diKryptera selezione di strategie
# Claude 
claude_agent_strategies = ['cfg_volume_contraction_bb']



# strategies = ['strategy_supertrend_breakout_trail', 'strategy_supertrend_adaptive_vol']
# strategies = strategy_registry(["supertrend*", "*breakout*"])
# strategies = strategy_registry(["supertrend*"])
# strategies = ['dpo_sroc', 'fvg', 'heikin_ashi']

# strategies = strategy_registry(["[b,c,d]*"])
# strategies = strategy_registry(["[h,i,l,m]*"])
# strategies = strategy_registry(["[l,m,n,o,p]*"])
# strategies = strategy_registry(["[m,t,u,v,z]*"])
# strategies = strategy_registry(["[a-z]*"])

# strategies = ['bollinger', 'tp_ma_crossover', 'ko_bb', 'zscore_mr_momo', 'tp_ma_crossover', 'zscore_mr_momo', 'bb_trend_following', 'zscore_mr_momo', 'macd_adx_sma', 'macd_tp_williams', 'cog_qqe', 'tp_ma_crossover', 'gmma', 'dualmom_breakout_spy', 'macd_bb', 'hma_atr', 'ichimoku_cloud', 'hma_atr', 'bb_trend_following']

# fixed_strategies = [
#     'cts_dpo', 'heikin_ashi', 'vzo_kvo', 'dma', 'ichimoku', 'tp_ma_crossover',
#     'ko_bb', 'sma_mf', 'cci_vortex', 'macd_williams', 'tema_heikin_ashi',
#     'macd_zero_line_rejection', 'ichimoku_cloud', 'gmma', 'trend_following',
#     'mean_reversion', 'bb_trend_following', 'bb_mean_reversion',
#     'hma_heikin_ashi', 'macd_daily_weekly', 'macd_mf', 'hma_atr',
#     'fractal_vortex', 'hma_stc', 'macd_aroon', 'ema_confluence', 'hma_smi',
#     'ma_slope_atr_rsi', 'kst_vri', 'macd_ravi', 'dpo_sroc', 'macd_bb',
#     'ema_trendline_projection', 'cmo', 'macd_tp_williams', 'cci_aroon',
#     'macd_cfo', 'simple_sma'
# ]
# strategies=fixed_strategies
# strategies = list(set(strategies)) # Uniq

# strategies = ['vol_regime_v2', 'pcr_percentile']

# strategies = ['qqq_trend_vol', 'qqq_dual_mom', 'qqq_ath_trail', 'spy_uptrend_dip', 'spy_quarter_switch', 'spy_ath_fastre', 'spy_adaptive_trail', 'spy_2of3_switch']
# strategies = strategy_registry(["spy*","qqq*"])

# strategies = kryptera_strategies
strategies = all_strategies


In [ ]:
#################################
# Selezione dei tickers
#################################

tickers = []
select_top_performers=False
company_data = {}

# - Liste semplici
# tickers = ["NVDA","LLY","AVGO","MSFT","XOM","SLB","ASML","SAP"]

tickers = greta_hy_tickers

# tickers=ai_dc_quantum_thematic

# Top performers

# USA: SP100
index = 'sp100'
rename_map={'BRK.B':'BRK-B'}
tickers_sp100 = extract_tickers_from_wikipedia(index,rename=rename_map)

# USA: nasdaq100
index = 'nasdaq100'
tickers_nasdaq100 = extract_tickers_from_wikipedia(index)

tickers_euro=stocks_euro

tickers_dict = {
    "sp100": tickers_sp100,
    "nasdaq100": tickers_nasdaq100,
    "euro": tickers_euro
}

exclude_tickers=["GOOG", "CSCO"]
year=2027
base_n_select=10
select_top_performers=True


In [ ]:
if select_top_performers:
    
    tickers, company_data, extra = build_top_momentum_universe(
        tickers_dict=tickers_dict,
        exclude_tickers=exclude_tickers,
        base_n_select=base_n_select,
        year=year,
    )
    company_info = company_data.loc[company_data.index.intersection(tickers)]
    my_display(company_info)
else:
    print(f"\n{BOLD}{len(tickers)}{RESET} tickers:\n\n{tickers}")

print(f"\n{BOLD}{len(strategies)}{RESET} strategie definite:\n\n{strategies}")
    


In [ ]:
print(f"tickers per l'anno {year}:\n")
sp100=extra['per_list_selected']['sp100']
nasdaq100=extra['per_list_selected']['nasdaq100']
euro=extra['per_list_selected']['euro']
print(f"sp100:      {" ".join(sp100)}")
print(f"nasdaq100:  {" ".join(nasdaq100)}")
print(f"euro:       {" ".join(euro)}")

# print(f"sp100:      {extra['per_list_selected']['sp100']}")
# print(f"nasdaq100:  {extra['per_list_selected']['nasdaq100']}")
# print(f"euro:       {extra['per_list_selected']['euro']}")
# print()
# display(tickers)

## Run strategy panel

In [ ]:
# Parametri per esecuzione massiva dei trading systems
risk_free_rate=0.02
init_cash=100_000.0
fees=0.001
slippage=0.002
price_col ='Open'
start_date="2015-01-01"
# end_date="2025-01-01"
end_date=now()

warmup_years=1
ratio="4:1"
selection_metric = "total_return" # 'recovery_factor' | callable(pf)->float 
# esempio callable: trade-off fra crescita e drawdown
# selection_metric=lambda pf: pf.cagr()*pf.sharpe_ratio() / abs(pf.max_drawdown() or 1e-9)
# selection_metric=lambda pf: pf.sharpe_ratio()

show_progress= True
verbose = False
save_results = True 
override = False
wfo_results_dir = _TSLAB_DEV_T_WFO_RESULTS_DIR

# wfo_results_dir = "WFO_TDEV_RESULTS"

In [ ]:
################################
# Suddivisione strategie
################################

# ELimina eventuali doppioni 
strategies = list(dict.fromkeys(strategies)) 
tickers = list(dict.fromkeys(tickers)) 

# 3 PC
# l=int(len(strategies)/3)

# strategies1=strategies[:l]
# print(strategies1)
# strategies2=strategies[l:l*2]
# print(strategies2)
# strategies3=strategies[l*2:]
# print(strategies3)


# 2 PC
l = max(1, int(len(strategies) / 2))
strategies1=strategies[:l]
print(strategies1)
strategies2=strategies[l:l*2]
if not strategies2: 
    strategies2=strategies1
print(strategies2)

################################
# Suddivisione tickers
################################

# 2 PC
l = max(1, int(len(tickers) / 2))
tickers1=tickers[:l]
print(tickers1)
tickers2=tickers[l:l*2]
if not tickers2: 
    tickers2=tickers1
print(tickers2)


In [ ]:
# # Top tickers per  l'anno 2026

# # sp100_top_tickers = ['LLY', 'GM', 'AMD', 'GOOGL', 'CAT', 'AMGN', 'C', 'COF', 'CSCO', 'MS'] # ['LLY'] -> B&H
# # dopo esclusione CSCO

# sp100_top_tickers = ['LLY', 'GM', 'AMD', 'GOOGL', 'CAT', 'AMGN', 'C', 'COF', 'MS', 'WFC']  # ['LLY'] -> B&H

# nasdaq100_top_tickers = ['MU', 'WDC', 'WBD', 'AMD', 'LRCX', 'GOOGL', 'AMAT', 'INSM', 'AZN', 'STX']

# euro_top_tickers = ['BAYN.DE', 'ACS.MC', 'MC.PA', 'ITX.MC', 'STLAM.MI', 'RWE.DE', 'SAN.MC', 'IBE.MC', 'MRK.DE', 'IFX.DE']

# top_tickers=sp100_top_tickers+nasdaq100_top_tickers+euro_top_tickers


In [ ]:
# Cosa testare e in che periodo storico

# tickers=["QQQ"]
# tickers=['MS']
# strategies=["cloud_lag_wr"]
# strategies=["qqq_ath_trail"]

strategies = all_strategies

strategies = ['k_gold_macd_reflex']
tickers = ["GOLD"]

%run ../../K-Strategy-Agent/strategies.ipynb

# Claude Agent 
strategies = ['cfg_volume_contraction_bb']
tickers = ["NVDA"]

# Risultati agente


# Strategie per US Trading
#
# ✅ Trading system candidati per il deploy:
# Ticker  Strategy                      Sharpe  CAR %   Total Return %MaxDD %  
# ─────────────────────────────────────────────────────────────────────────────
# AMAT    aee_qke                       1.06    N/A     486.63        50.01    
# COP     adbe_efficiency_momentum      0.61    N/A     109.33        64.41    
# FDX     aee_qke                       0.90    N/A     255.15        49.61    

# ======================================================================
#   RIEPILOGO TEST STRATEGIE
# ======================================================================
#   Combinazioni testate : 180  (10 ticker × 18 strategie)
#   Precheck superato    : 10
#   Scartate (precheck)  : 170

#   Strategie che hanno completato WFO:

#   Ticker  Strategy                      Sharpe  CAR %   Total Return %MaxDD %  
#   -----------------------------------------------------------------------------
#   INTC    bkr_er_rsi                    0.03    N/A     -12.08        50.85    
#   INTC    cdns_bollinger_macd           0.33    N/A     13.90         70.18    
#   INTC    cop_demarker_supertrend       0.55    N/A     91.26         60.78    
#   LRCX    trmb_sma_wma                  1.16    N/A     671.84        56.85    
#   FDX     cop_demarker_supertrend       0.81    N/A     188.90        58.48    
#   AVGO    trmb_sma_wma                  1.51    N/A     1374.90       47.16    
#   COP     hma_bb_crossover              0.58    N/A     101.89        65.29    
#   COP     trmb_sma_wma                  0.80    N/A     224.95        62.85    
#   COP     cop_demarker_supertrend       0.90    N/A     294.30        49.35    
#   DE      trmb_sma_wma                  0.92    N/A     220.40        38.02    

# ======================================================================
# 2026-04-20 11:58:09  INFO      ═══ Test strategie completato ═══

# ======================================================================
#   RIEPILOGO TEST STRATEGIE
# ======================================================================
#   Combinazioni testate : 180  (10 ticker × 18 strategie)
#   Precheck superato    : 29
#   Scartate (precheck)  : 151

#   Strategie che hanno completato WFO:

#   Ticker  Strategy                      Sharpe  CAR %   Total Return %MaxDD %  
#   -----------------------------------------------------------------------------
#   ENI.MI  pstg_roc_vidya                0.72    N/A     108.27        51.59    
#   ACS.MC  cfg_volume_contraction_bb     1.11    N/A     366.52        51.36    
#   REP.MC  cms_keltner_srpr              0.87    N/A     191.68        45.87    
#   REP.MC  msi_bb_qi                     0.71    N/A     90.57         38.36    
#   PRY.MI  aee_qke                       1.28    N/A     534.61        47.23    
#   AI.PA   pstg_roc_vidya                0.72    N/A     70.35         23.42    
#   AI.PA   cdns_bollinger_macd           1.09    N/A     157.71        23.24    
#   IFX.DE  adbe_efficiency_momentum      0.60    N/A     114.52        49.96    
#   IFX.DE  cdns_bollinger_macd           0.87    N/A     287.15        50.21    
#   BAS.DE  ko_trend_strength_bollinger_reversion0.41    N/A     39.77         42.46    
#   BAS.DE  pstg_roc_vidya                0.40    N/A     39.15         42.46    
#   BAS.DE  adbe_efficiency_momentum      0.40    N/A     41.41         42.46    
#   BAS.DE  smci_momentum_vidya           -0.21   N/A     -27.91        41.26    
#   BAS.DE  bkr_er_rsi                    0.95    N/A     102.88        19.92    
#   BAS.DE  msi_bb_qi                     0.14    N/A     3.58          39.25    
#   TEF.MC  ko_trend_strength_bollinger_reversion-0.14   N/A     -22.77        39.46    
#   TEF.MC  cms_keltner_srpr              0.19    N/A     5.75          42.41    
#   TEF.MC  cdns_bbmacd                   0.35    N/A     2.94          2.58     
#   TEF.MC  cme_qqe_bears                 0.10    N/A     2.38          34.52    
#   TEF.MC  pstg_roc_vidya                0.30    N/A     21.09         47.72    
#   TEF.MC  smci_momentum_vidya           -0.25   N/A     -28.53        42.90    
#   TEF.MC  connors_bollinger_reversion   inf     N/A     0.00          nan      
#   TEF.MC  bkr_er_rsi                    0.71    N/A     56.50         26.17    
#   TEF.MC  msi_bb_qi                     -0.32   N/A     -27.05        36.23    
#   CLNX.MC cfg_volume_contraction_bb     0.25    N/A     11.63         49.96    
#   CLNX.MC ko_trend_strength_bollinger_reversion0.43    N/A     49.07         48.92    
#   CLNX.MC adbe_efficiency_momentum      0.12    N/A     -9.12         55.79    
#   G.MI    aee_qke                       1.30    N/A     239.12        30.92    
#   G.MI    cop_demarker_supertrend       1.26    N/A     240.36        26.86    

# ======================================================================
# 2026-04-20 19:20:29  INFO      ═══ Test strategie completato ═══


# EURO
# ======================================================================
#   RIEPILOGO TEST STRATEGIE
# ======================================================================
#   Combinazioni testate : 220  (10 ticker × 22 strategie)
#   Precheck superato    : 7
#   Scartate (precheck)  : 213

#   Strategie che hanno completato WFO:

#   Ticker  Strategy                      Sharpe  CAR %   Total Return %MaxDD %  
#   -----------------------------------------------------------------------------
#   REP.MC  are_dem_kc                    0.91    N/A     225.97        44.18    
#   IFX.DE  mpwr_bb_std                   0.60    N/A     120.77        52.01    
#   IFX.DE  bb_expansion                  1.00    N/A     380.51        41.11    
#   BAS.DE  are_dem_kc                    0.53    N/A     71.54         35.30    
#   TEF.MC  bb_expansion                  0.44    N/A     46.86         39.23    
#   CLNX.MC mpwr_bb_std                   0.11    N/A     -11.07        56.66    
#   CLNX.MC bb_expansion                  0.02    N/A     -22.83        58.18    

# ======================================================================
# 2026-04-21 13:23:57  INFO        Risultati Excel aggiornati: /l/disc1/Insync/lf27963@gmail.com/Google Drive/Colab Notebooks/Portafogli/Sviluppo/ChatGpt/TSlab_project/outputs/WFO_T_DEV_RESULTS/test_results_history.xlsx  (7 righe totali)
# 2026-04-21 13:23:57  INFO      ═══ Test strategie completato ═══

start_date="2015-01-01"
# end_date="2025-01-01"
end_date = now()


In [ ]:
# New panel
# %run _bootstrap_dev.ipynb

override=True
use_catalog=False
# use_catalog=True

return_catalog=True
return_joblist=True
dry_run=False
verbose=True
show_progress=True


ratio="4:1"
price_col="Open"
selection_metric="total_return"

# Filtri di overfitting

'''
    precheck_require_recommend_wfo : bool, default False
        Definisce quale logica usare per decidere il pass/fail del precheck stress.

        - Se True:
          il passaggio dipende dalla decisione sintetica `recommend_wfo` prodotta
          dall'analisi overfitting.

        - Se False:
          la decisione NON usa direttamente `recommend_wfo`; se
          `precheck_use_equity_gate=True`, il task passa solo se la metrica
          `beat_bh_pct_by_total_return` supera la soglia
          `precheck_min_beat_bh_pct`.

'''
# Combinazione piu' efficace (versione prededente)
# precheck_require_recommend_wfo = False
# precheck_mode = "stress" 
# # precheck_mode = None 
# precheck_use_equity_gate=True
# precheck_min_beat_bh_pct=0.50


# Combinazione piu' efficace
precheck_mode = "stress"
precheck_require_best_is_beat_bh = True
precheck_min_beat_bh_pct = 0.50
precheck_min_best_is_excess = 0.0
precheck_use_equity_gate = False
precheck_require_recommend_wfo = True
precheck_stress_n_samples=200
precheck_stress_method="lhs"
precheck_seed=42
cache_precheck=False

# New: parametro scenario
#
# scenario="A"  Esplorazione
#     Gate1 ON  (best param batte B&H)
#     Gate2 OFF
#     Gate3 OFF
#     → Permissivo. Passa quasi tutto. Utile per primo screening.

# scenario="B"  Qualità base  ← consigliato per strategie generate dall'agente
#     Gate1 ON  (best param batte B&H)
#     Gate2 ON  (beat_bh_pct >= 40%)
#     Gate3 OFF
#     → Richiede un minimo di robustezza parametrica.

# scenario="C"  Qualità media
#     Gate1 ON  (best param batte B&H)
#     Gate2 ON  (beat_bh_pct >= 50%)
#     Gate3 OFF
#     → Bilanciato. Default consigliato per test sistematici.

# scenario="D"  Configurazione notebook
#     Gate1 ON  (best param batte B&H)
#     Gate2 OFF
#     Gate3 ON  (recommend_wfo=True)
#     → Selettivo sul giudizio composito, non sulla robustezza diretta.

# scenario="E"  Produzione
#     Gate1 ON  (best param batte B&H)
#     Gate2 ON  (beat_bh_pct >= 50%)
#     Gate3 ON  (recommend_wfo=True)
#     → Tutti i gate. Solo per strategie candidate al deploy.


df_panel, results_panel, extra = wfo_strategy_panel(
    tickers=tickers,
    strategies=strategies,
    start_date=start_date,
    end_date=end_date,
    ratio=ratio,
    price_col=price_col,
    selection_metric=selection_metric,
    warmup_years=1,
    show_progress=show_progress,
    verbose=verbose,
    save_results=save_results,         # nel test, evita IO su disco
    wfo_results_dir=wfo_results_dir,
    override=override,
    # catalog
    use_catalog=use_catalog,
    # scenario
    scenario="E",
    # Precheck (usati solo se scenario=None)
    precheck_require_best_is_beat_bh=precheck_require_best_is_beat_bh,
    precheck_min_best_is_excess = precheck_min_best_is_excess,
    precheck_mode=precheck_mode,
    precheck_stress_n_samples=precheck_stress_n_samples,
    precheck_stress_method=precheck_stress_method,
    precheck_seed=precheck_seed,
    precheck_use_equity_gate=precheck_use_equity_gate,
    precheck_min_beat_bh_pct=precheck_min_beat_bh_pct,
    precheck_require_recommend_wfo=precheck_require_recommend_wfo,
    # caching precheck OFF nel test (non serve)
    cache_precheck=cache_precheck,
)



## Analisi risultati

In [ ]:
# %run _bootstrap_dev.ipynb

include_best_params = True

selection="winners" #  None | "best_return" | "best_dd" | "best_ratio"
# selection="best_dd"     
# selection="best_return" 
# selection=None          

summary_df, df_wfo_all, noncompliant_paths, winners_list, styler = wfo_strategy_panner(
    # strategies=['macd_ravi', 'bollinger'],
    strategies=['kvo_zero_line'],
    # strategies=strategies, # analisi delle strategie ultimo run
    # strategies=['bb_trend_following'],
    # strategies=['stdch_momvol', 'hma_atr', 'supertrend_adaptive_vol'],
    # tickers=tickers,
    # tickers=['SPY', 'QQQ'],
    # tickers=['ENI.MI'],
    # tickers=['WFC'],
    tickers=["CARR"],
    # strategies=["qqq_ath_trail"],   # <-- metti una strategia che sai esistere
    # ratios=ratio,
    # tickers=["QQQ"],
    # strategies=["qqq_ath_trail"],   # <-- metti una strategia che sai esistere
    selection=selection,
    wfo_results_dir=wfo_results_dir,
    include_best_params=include_best_params,
    # debug=False
)


In [ ]:

results_summary = analyze_wfo_results(
    winners_list=winners_list,
    # tickers=tickers if "tickers" in locals() else None,
    tickers=tickers,
    # tickers=['WFC'],
    # tickers=nasdaq100_tickers_no_strategy,
    # tickers=sp100_tickers_no_strategy,
    # selection=selection,
    top_n=10
)


In [ ]:
#
# Analisi dei risultati
#

winner_strategies = [w[0] for w in winners_list]
winner_strategies = list(set(winner_strategies)) # Uniq

# winner_tickers
strategy_tickers = [w[1] for w in winners_list]
strategy_tickers = list(set(strategy_tickers)) # Uniq


print(f"{BOLD}{GREEN}{len(winner_strategies)}{RESET} strategie vincenti con selettore {BOLD}{selection}{RESET}:\n\n{winner_strategies}")
print("")

print(f"{BOLD}{GREEN}{len(strategy_tickers)}{RESET} tickers hanno una strategia vincente:\n\n{strategy_tickers}")
print("")

# # b^h tickers
if "tickers" in locals():
    bh_tickers = [x for x in tickers if x not in strategy_tickers]
    print(f"{BOLD}{RED}{len(bh_tickers)}{RESET} tickers non hanno una strategia vincente:\n\n{bh_tickers}")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm, skew, kurtosis
from typing import Union


def deflated_sharpe_ratio(
    sharpe_ratio, n_obs, skewness, kurtosis, n_trials, sr_benchmark=0.0
):
    sr_std = np.sqrt(
        (1 - skewness * sharpe_ratio + ((kurtosis - 1) / 4) * sharpe_ratio**2)
        / (n_obs - 1)
    )
    if n_trials <= 1:
        sr_star = sr_benchmark
    else:
        euler_mascheroni = 0.5772156649
        z = (1 - euler_mascheroni) * norm.ppf(1 - 1 / n_trials) + \
            euler_mascheroni * norm.ppf(1 - 1 / (n_trials * np.e))
        sr_star = sr_benchmark + sr_std * z

    dsr = norm.cdf(
        (sharpe_ratio - sr_star) * np.sqrt(n_obs - 1) /
        np.sqrt(1 - skewness * sharpe_ratio + ((kurtosis - 1) / 4) * sharpe_ratio**2)
    )
    return {
        "dsr": round(dsr, 6),
        "sr_star": round(sr_star, 6),
        "sr_std": round(sr_std, 6),
        "is_significant": dsr > 0.95,
    }


def compute_dsr_from_returns(
    returns: Union[pd.Series, np.ndarray, list],
    n_trials: int = 1,
    periods_per_year: int = 252,
    sr_benchmark: float = 0.0,
) -> dict:
    from scipy.stats import skew as _skew, kurtosis as _kurt
    r = np.asarray(returns, dtype=float)
    r = r[~np.isnan(r)]
    n_obs  = len(r)
    mean   = np.mean(r)
    std    = np.std(r, ddof=1)
    skew_  = _skew(r)
    kurt_  = _kurt(r, fisher=True)
    sr     = (mean / std) * np.sqrt(periods_per_year) if std > 0 else 0.0

    return deflated_sharpe_ratio(
        sharpe_ratio=sr,
        n_obs=n_obs,
        skewness=skew_,
        kurtosis=kurt_,
        n_trials=n_trials,
        sr_benchmark=sr_benchmark,
    )

def compute_panel_dsr(
    results_panel: dict,
    periods_per_year: int = 252,
    sr_benchmark: float = 0.0,
) -> pd.DataFrame:
    """
    Calcola il DSR per tutte le combinazioni (Ticker, Strategy) in results_panel.

    Parameters
    ----------
    results_panel    : dict output di wfo_strategy_panel_NEW
                       chiavi: (ticker, strategy) | valori: dict con 'portfolio'
    periods_per_year : 252 giornaliero | 52 settimanale | 12 mensile
    sr_benchmark     : SR minimo accettabile (default 0)

    Returns
    -------
    pd.DataFrame con colonne:
        Ticker, Strategy, n_obs, sharpe_ratio, dsr, sr_star, sr_std, is_significant
    """
    n_trials = len(results_panel)  # ogni (ticker, strategy) è un trial distinto
    rows = []

    for (ticker, strategy), val in results_panel.items():
        try:
            returns = val['portfolio'].returns().dropna().values

            if len(returns) < 10:
                rows.append({
                    "Ticker": ticker, "Strategy": strategy,
                    "n_obs": len(returns), "sharpe_ratio": np.nan,
                    "dsr": np.nan, "sr_star": np.nan,
                    "sr_std": np.nan, "is_significant": False,
                    "error": "insufficient data",
                })
                continue

            result = compute_dsr_from_returns(
                returns=returns,
                n_trials=n_trials,
                periods_per_year=periods_per_year,
                sr_benchmark=sr_benchmark,
            )

            # Sharpe annualizzato grezzo per riferimento
            r = returns
            sr_raw = (np.mean(r) / np.std(r, ddof=1)) * np.sqrt(periods_per_year)

            rows.append({
                "Ticker":         ticker,
                "Strategy":       strategy,
                "n_obs":          len(returns),
                "sharpe_ratio":   round(sr_raw, 6),
                "dsr":            result["dsr"],
                "sr_star":        result["sr_star"],
                "sr_std":         result["sr_std"],
                "is_significant": result["is_significant"],
                "error":          None,
            })

        except Exception as e:
            rows.append({
                "Ticker": ticker, "Strategy": strategy,
                "n_obs": None, "sharpe_ratio": np.nan,
                "dsr": np.nan, "sr_star": np.nan,
                "sr_std": np.nan, "is_significant": False,
                "error": str(e),
            })

    df = pd.DataFrame(rows).sort_values("dsr", ascending=False).reset_index(drop=True)
    return df

In [ ]:
# ATTENZIONE: 
# Il DSR aggiunge valore reale nel tuo caso solo quando lanci il panel su molte strategie e ticker contemporaneamente e vuoi scegliere il winner tra tutti 
# Se applicato a 1 strategia e 1 ticker il DSR coincide con lo sharpe WFO e non aggiunge informaioni utili.

dsr_df = compute_panel_dsr(results_panel, periods_per_year=252)

my_display(dsr_df[["Ticker", "Strategy", "sharpe_ratio", "dsr", "sr_star", "is_significant"]],title="DSR per tutte le combinazioni")



In [ ]:
# Analisi path non compliant
# dry_run=True
# deleted, failed = delete_paths(noncompliant_paths, dry_run=dry_run)   # preview
# print (f"Rimossi i seguenti files: {deleted}") if not dry_run else (f"I seguenti files: {deleted} sono da rimuovere")


## Analisi Trading System

In [ ]:
# strategy = 'macd_bb'
# symbol = 'ACX.MC'
# symbol = 'UCG.MI'

# strategy = 'bb_trend_following'	
# symbol = 'LDO.MI'

# strategy = 'hma_atr'
# symbol = 'BNP.PA'

# strategy = 'ko_bb'
# symbol = 'ENI.MI'

# strategy = 'vol_regime'
# symbol = 'SAN.MC'


# strategy = 'macd_adx_sma'

# symbol = 'CLSK'

# strategy = 'macd_tp_williams'

# strategy = 'ichimoku_cloud'

# symbol = 'MARA'

# strategy = 'spy_2of3_switch'
# symbol = 'NQSE.DE'

# strategy = 'spy_ath_fastre'
# symbol = 'ENEL.MI'


# strategy = 'ko_bb'
# symbol = 'ENI.MI'


# strategy = 'bb_mean_reversion'
# symbol = 'RWE.DE'

# strategy='macd_bb'
# symbol='ACX.MC'

# strategy = 'ko_cci'
# symbol = 'AMAT'

# strategy = 'zscore_mr_momo'   # OK CAGR	23.37% Max Drawdown	32.72%
# strategy = 'qqq_ath_trail'    # OK CAGR	20.40% Max Drawdown	19.00%
# strategy = 'spy_2of3_switch'  # OK CAGR	20.95% Max Drawdown	27.56%
# symbol = 'QQQ'

# strategy = 'spy_black_swan'
# symbol = 'SPY'

# strategy = 'bb_trend_following'	
# strategy = 'stdch_momvol'	
# strategy = 'hma_atr'
# symbol = 'WFC'

# strategy = 'momentum_rank1d'	
# symbol = 'COF'

# strategy="qqq_ath_trail"  
# symbol="QQQ"

# wfo_results_dir = "./outpus/WFO_TDEV_RESULTS"

strategy = "kvo_zero_line"    
symbol   = "CARR"

portfolio, bh_portfolio, wfo_results = load_ts(symbol=symbol,
        strategy=strategy,
         # ratio=ratio,                                      
        wfo_results_dir=wfo_results_dir)


In [ ]:
# figs = generate_portfolio_performance(pf=portfolio,
#                                        portfolio_title=symbol,
#                                        pf_b_h=portfolio_bh,
#                                        portfolio_ts= None,
#                                        benchmark=None)

# %run _bootstrap_dev.ipynb

auto_adjust = True # tipicamente nel calcolo performance uso prezi rettificati per ottenere Total Return. Se False -> Price return

portfolio_title = f"{symbol} - Total"
portfolio_title += " Return" if auto_adjust else " Price" 


figs = generate_portfolio_performance(
    pf=portfolio,
    portfolio_title=portfolio_title,
    benchmark="Internal Benchmark (B&H)",
    alpha_analysis=True,
    show_plots=True
)

## Montecarlo, ovvero questo risultato OOS è robusto o è ancora fragile / lucky?

In [ ]:
#
# Esegue tutti i metodi Montecarlo definiti
# per verificare:
# - stabilità
# - dispersione dei risultati
# - probabilità di worst-case

%run _bootstrap_dev.ipynb

# Parametri di base
n_simulations = 10_000
show_plots = True
show_summary = True
slippage = 0.0001           # -0.01% al giorno
shock_frequency = 0.002     # 0.2% dei giorni con eventi negativi
shock_magnitude = (0.05, 0.15)
init_cash = 100_000

# Returns del TS e benchmark dal B&H vectorbt
ts_returns = portfolio.returns()          # Serie dei rendimenti giornalieri del TS
benchmark_return = float(round(bh_portfolio.total_return(), 4))
benchmark_drawdown = abs(float(round(bh_portfolio.max_drawdown(), 4)))  # assumo che max_drawdown sia negativo

methods_results, summary_df = run_all_mc_methods(
    portfolio_returns=ts_returns,
    init_value=init_cash,
    benchmark_return=benchmark_return,
    benchmark_drawdown=benchmark_drawdown,
    n_simulations=n_simulations,
    seed=42,
    show_method_plots=True,        # se vuoi i plot anche per il base (e per gli altri)
    show_method_summaries=True,
    block_size=10,
    regime_window=20,
)


In [ ]:
# 1) dopo run_all_mc_methods(...) hai methods_results
mc_compare_df = compare_mc_methods(
    methods_results=methods_results,
    benchmark_return=benchmark_return,
    benchmark_drawdown=benchmark_drawdown
)
my_display(mc_compare_df,"===== MC COMPARE (esteso) =====")


# 2) decisione deploy (portfolio-aware)
deploy_report = mc_deploy_recommendation(
    mc_compare_df=mc_compare_df,
    benchmark_return=benchmark_return,
    benchmark_drawdown=benchmark_drawdown,
    portfolio_mode=True,
    portfolio_ts_count=10,
    expected_weight=0.10,   # se prevedi ~equal weight; se non lo sai, ometti e usa 1/10
    rules={
        # soglie di default già ok, ma puoi renderle più severe:
        # "min_p_dom": 0.60,
        # "max_p_fail": 0.15,
        # "min_tail_return_5": -0.15,
        # "max_tail_dd_95": 0.55,
    }
)
print()
print("="*30) 
print("DEPLOY DECISION")
print("="*30)
print()
print(f"Decision : {BOLD}{deploy_report['decision']}{RESET}")
print(f"Score    : {BOLD}{deploy_report['score']:.1f}/100{RESET}\n")

print("Motivazioni:")
for r in deploy_report["reasons"]:
    print(f" - {r}")

print("\nDettaglio regole:")
for k, v in deploy_report["rule_checks"].items():
    try:
        val = v["value"]
    except:
        continue
    thr = v["thr"]
    ok  = v["ok"]
    if val is None:
        print(f" - {k}: skipped (benchmark non disponibile)")
    else:
        print(f" - {k}: value={val:.3f} thr={thr:.3f} -> {BOLD}{'\033[32mOK' if ok else '\033[31mFAIL'}{RESET}")

## Monte Carlo Docs

### 🎯 Obiettivo del Monte Carlo nel framework

Nel framework (WFO + filtro overfitting + Monte Carlo), le metodologie Monte Carlo hanno un ruolo molto preciso: **validare la robustezza statistica OOS** della strategia, stressando la distribuzione dei risultati.

Ti sintetizzo le principali tecniche utilizzate (coerenti con il tuo approccio quantitativo).

---


Dopo:

1. **Backtest / IS fit**
2. **Walk-Forward (OOS reale)**

il Monte Carlo serve a rispondere a:
👉 *“Questo risultato OOS è robusto o è ancora fragile / lucky?”*

In altre parole:

* stabilità
* dispersione dei risultati
* probabilità di worst-case

---

# 🧪 1. Monte Carlo su Trade (Shuffle / Bootstrap semplice)

### Logica

* Prendi la serie dei trade OOS
* Li **mescoli (shuffle)** o campioni con replacement
* Ricostruisci N equity curve

### Cosa testa

* **dipendenza dall’ordine dei trade**
* fragilità del path (path dependency)

### Metriche tipiche

* distribuzione CAGR
* max drawdown distribution
* percentili (5%, 50%, 95%)

### Interpretazione

* se degrada molto → strategia instabile
* se regge → robusta

---

# 🧪 2. Block Bootstrap (la più importante nel tuo framework)

### Logica

* invece di singoli trade → campioni **blocchi di rendimenti consecutivi**
* preservi autocorrelazione e clustering di volatilità

👉 coerente con il fatto che i mercati **non sono i.i.d.**

### Variante

* fixed block
* random block length

### Cosa testa

* resilienza a:

  * regimi di mercato
  * sequenze negative
  * clustering di drawdown

### Perché è centrale

Nel tuo framework è la tecnica più corretta perché:

* lavori su strategie dinamiche (WFO)
* i rendimenti NON sono indipendenti

---

# 🧪 3. Monte Carlo su Equity (resampling rendimenti)

### Logica

* lavori sui **returns bar-by-bar**
* resampling (bootstrap o shuffle)
* ricostruisci equity curve

### Differenza vs trade MC

* più fine (time series)
* meno legato alla logica operativa

### Uso

* utile per strategie ad alta frequenza
* meno interpretabile per TS discrezionali

---

# 🧪 4. Noise Injection (perturbazione)

### Logica

* aggiungi rumore ai rendimenti:

  * slippage variabile
  * execution noise
  * micro-shock sui prezzi

### Cosa testa

* robustezza a:

  * costi reali
  * imperfezioni di mercato

### Esempi

* ±X% sui returns
* widening spread

---

# 🧪 5. Parameter Perturbation Monte Carlo

### Logica

* non usi solo i parametri ottimi WFO
* li perturbi localmente

### Cosa testa

* stabilità del modello rispetto ai parametri

👉 è il ponte tra:

* overfitting test
* Monte Carlo

---

# 🧪 6. Walk-Forward Monte Carlo (avanzato)

### Logica

* ripeti la WFO su:

  * dataset perturbati
  * subset temporali
  * bootstrapped series

### Cosa testa

* stabilità dell’intero processo:

  * selezione parametri
  * regime adaptation

---

# 📊 Output tipici del Monte Carlo

Nel framework dovresti avere:

### Distribuzioni

* CAGR distribution
* Sharpe distribution
* Max DD distribution

### Risk metrics

* probabilità di perdita
* worst-case percentile
* skewness / fat tails

### Robustness score

(esempio)

* % simulazioni profittevoli
* stability ratio

---

# 🔗 Collegamento con il tuo processo

Pipeline corretta:

### 1️⃣ Overfitting filter

* Deflated Sharpe Ratio
* selezione strategie robuste

### 2️⃣ Walk Forward

* performance reale OOS

### 3️⃣ Monte Carlo

* stress test della OOS
* validazione probabilistica

👉 Questo è coerente anche con la filosofia della dispensa:

* il backtest non basta
* serve validazione rigorosa per evitare risultati casuali 

---

# ⚠️ Insight critico (importante)

Errore comune:

> usare Monte Carlo per “migliorare” la strategia

Nel tuo framework:
👉 Monte Carlo è **solo validazione**, non ottimizzazione

Se lo usi per selezionare strategie → reintroduci overfitting.

---

# 🧠 Sintesi operativa

Le 3 tecniche chiave nel tuo contesto sono:

1. **Block Bootstrap (core)**
2. **Trade Shuffle**
3. **Parameter Perturbation**

Le altre sono complementari.

---

Se vuoi, nel prossimo step posso:

* formalizzare **una funzione standard Monte Carlo** compatibile con vectorbt/WFO
* oppure definire un **robustness score unico** da integrare nella pipeline (post-WFO)


## Company Info

In [ ]:
# # 
# # Company info
# #
sp100_top_tickers = ['LLY', 'GM', 'AMD', 'GOOGL', 'CAT', 'AMGN', 'C', 'COF', 'CSCO', 'MS']
nasdaq100_top_tickers = ['MU', 'WDC', 'WBD', 'AMD', 'LRCX', 'GOOGL', 'AMAT', 'INSM', 'AZN', 'STX']
euro_top_tickers = ['BAYN.DE', 'ACS.MC', 'MC.PA', 'ITX.MC', 'STLAM.MI', 'RWE.DE', 'SAN.MC', 'IBE.MC', 'MRK.DE', 'IFX.DE']

tickers = euro_top_tickers
_, company_data = fetch_data_and_companies(tickers, start_date, end_date)

company_info = company_data.loc[company_data.index.intersection(tickers)]
my_display(company_info)